# Web y APIs · Flask

Aplicaciones web y APIs con `flask`. En un proyecto real se arrancan con `app.run()` o `flask run`; aquí se prueban con el cliente de pruebas de Flask (`app.test_client()`), que hace peticiones sin levantar un servidor.

## 1. Acortador de enlaces persistente

El acortador guarda en `url_map` la URL larga de cada URL corta (los 7 primeros caracteres de su SHA-256). Usar `sqlite3` para que las URL acortadas no se pierdan al reiniciar la aplicación.

In [1]:
import hashlib
import sqlite3
from flask import Flask, redirect, request

def crea_acortador(ruta_bd):
    app = Flask(__name__)
    with sqlite3.connect(ruta_bd) as con:
        con.execute("CREATE TABLE IF NOT EXISTS urls (corta TEXT PRIMARY KEY, larga TEXT)")

    @app.route("/acorta", methods=["POST"])
    def acorta():
        larga = request.form["url"]
        corta = hashlib.sha256(larga.encode()).hexdigest()[:7]
        with sqlite3.connect(ruta_bd) as con:
            con.execute("INSERT OR REPLACE INTO urls VALUES (?, ?)", (corta, larga))
        return f"Acortada: {larga} -> {corta}"

    @app.route("/<corta>")
    def redirige(corta):
        with sqlite3.connect(ruta_bd) as con:
            fila = con.execute("SELECT larga FROM urls WHERE corta = ?", (corta,)).fetchone()
        return redirect(fila[0]) if fila else ("No encontrada!", 404)

    return app

cliente = crea_acortador("urls.db").test_client()
print(cliente.post("/acorta", data={"url": "http://www.google.com"}).text)
# una aplicación nueva lee la misma base de datos: simula un reinicio
reiniciada = crea_acortador("urls.db").test_client()
r = reiniciada.get("/253d142")
print(r.status_code, r.headers["Location"], reiniciada.get("/nada").status_code)

Acortada: http://www.google.com -> 253d142
302 http://www.google.com 404


## 2. API de conversión de moneda

La API convierte euros a dólares en `/to-usd/<eur>`. Añadir `/to-eur/<usd>`, aceptar también `/to-usd?eur=100` y un *endpoint* para actualizar el tipo de cambio. Se usa `PUT` porque actualizar el recurso con el mismo valor varias veces deja el mismo resultado (es idempotente), a diferencia de `POST`.

In [2]:
from flask import jsonify

app = Flask(__name__)
app.config["EUR_A_USD"] = 1.1

def convierte(valor, factor, origen, destino):
    try:
        valor = float(valor)
    except (TypeError, ValueError) as e:
        return jsonify({"error": str(e)}), 400
    return jsonify({origen: valor, destino: round(valor * factor, 4)})

@app.route("/to-usd/<eur>")
@app.route("/to-usd")
def to_usd(eur=None):
    return convierte(eur if eur is not None else request.args.get("eur"), app.config["EUR_A_USD"], "EUR", "USD")

@app.route("/to-eur/<usd>")
def to_eur(usd):
    return convierte(usd, 1 / app.config["EUR_A_USD"], "USD", "EUR")

@app.route("/ratio", methods=["PUT"])
def actualiza_ratio():
    app.config["EUR_A_USD"] = float(request.get_json()["EUR_A_USD"])
    return jsonify({"EUR_A_USD": app.config["EUR_A_USD"]})

c = app.test_client()
print(c.get("/to-usd/100").json, c.get("/to-usd?eur=100").json, c.get("/to-eur/110").json, c.get("/to-usd/abc").status_code)
print(c.put("/ratio", json={"EUR_A_USD": 1.2}).json, c.get("/to-usd/100").json)

{'EUR': 100.0, 'USD': 110.0} {'EUR': 100.0, 'USD': 110.0} {'EUR': 100.0, 'USD': 110.0} 400
{'EUR_A_USD': 1.2} {'EUR': 100.0, 'USD': 120.0}


## 3. Analizador de textos

Construir la parte de análisis de textos de una aplicación web: un formulario envía un texto y la respuesta muestra el número de caracteres, palabras y frases y las palabras más frecuentes.

In [3]:
import re
from collections import Counter
from flask import render_template_string

PAGINA = """<h1>Analizador de textos</h1>
<form method="post"><textarea name="texto"></textarea><button>Analizar</button></form>
{% if resultado %}<ul>{% for clave, valor in resultado.items() %}<li>{{ clave }}: {{ valor }}</li>{% endfor %}</ul>{% endif %}"""

app = Flask(__name__)

def analiza(texto):
    palabras = re.findall(r"\w+", texto.lower())
    return {"caracteres": len(texto), "palabras": len(palabras), "frases": len(re.findall(r"[.!?]+", texto)),
            "más frecuentes": Counter(palabras).most_common(3)}

@app.route("/analizador", methods=["GET", "POST"])
def analizador():
    resultado = analiza(request.form["texto"]) if request.method == "POST" else None
    return render_template_string(PAGINA, resultado=resultado)

c = app.test_client()
print(c.get("/analizador").status_code)
print(c.post("/analizador", data={"texto": "Python es genial. Python es fácil. ¡Aprende Python!"}).text.split("<ul>")[1])

200
<li>caracteres: 51</li><li>palabras: 8</li><li>frases: 3</li><li>más frecuentes: [(&#39;python&#39;, 3), (&#39;es&#39;, 2), (&#39;genial&#39;, 1)]</li></ul>


## 4. API de estudiantes con MongoDB

API REST para los estudiantes de *30 Days Of Python* (nombre, país, ciudad, nacimiento, habilidades y biografía) con las operaciones `GET` (todos y uno), `POST`, `PUT` y `DELETE` sobre MongoDB. Para no necesitar un servidor de MongoDB, se usa `mongomock`, que imita la API de `pymongo` (en producción bastaría con `pymongo.MongoClient(URI)`).

In [4]:
import mongomock
from bson.objectid import ObjectId

app = Flask(__name__)
db = mongomock.MongoClient().thirty_days_of_python

def a_json(estudiante):
    return {**estudiante, "_id": str(estudiante["_id"])}

@app.route("/api/v1.0/students", methods=["GET"])
def students():
    return jsonify([a_json(s) for s in db.students.find()])

@app.route("/api/v1.0/students/<id>", methods=["GET"])
def single_student(id):
    s = db.students.find_one({"_id": ObjectId(id)})
    return (jsonify(a_json(s)), 200) if s else (jsonify({"error": "no encontrado"}), 404)

@app.route("/api/v1.0/students", methods=["POST"])
def create_student():
    nuevo = {k: request.json[k] for k in ("name", "country", "city", "birthyear", "skills", "bio")}
    return jsonify({"_id": str(db.students.insert_one(nuevo).inserted_id)}), 201

@app.route("/api/v1.0/students/<id>", methods=["PUT"])
def update_student(id):
    db.students.update_one({"_id": ObjectId(id)}, {"$set": request.json})
    return single_student(id)

@app.route("/api/v1.0/students/<id>", methods=["DELETE"])
def delete_student(id):
    borrados = db.students.delete_one({"_id": ObjectId(id)}).deleted_count
    return jsonify({"borrados": borrados})

c = app.test_client()
nuevo = {"name": "Asabeneh", "country": "Finland", "city": "Helsinki", "birthyear": 1250, "skills": ["Python"], "bio": "Teacher"}
id_ = c.post("/api/v1.0/students", json=nuevo).json["_id"]
c.post("/api/v1.0/students", json={**nuevo, "name": "Lidiya", "city": "Espoo"})
print(len(c.get("/api/v1.0/students").json))
print(c.put(f"/api/v1.0/students/{id_}", json={"skills": ["Python", "Flask"]}).json["skills"])
print(c.delete(f"/api/v1.0/students/{id_}").json, c.get(f"/api/v1.0/students/{id_}").status_code)

2
['Python', 'Flask']
{'borrados': 1} 404
